# Fix Count Files

This notebook rebuilds the downstream count files from the existing parquet files on disk using the corrected combination-level positive export and the fixed background normalization logic.

In [1]:
%config InlineBackend.figure_format = 'retina'

import pandas as pd

from amr.filtering import build_positive_dataset
from amr.normalisation import BackgroundNormaliseConfig, normalise_by_background

## Load existing files from disk

In [2]:
pmid_df = pd.read_parquet("pmids_results.parquet")
filtered_df = pd.read_parquet("pmids_results_filtered.parquet")
pa_counts_df = pd.read_parquet("pathogen_tiab_counts.parquet")
ab_counts_df = pd.read_parquet("antibiotic_tiab_counts.parquet")

print("Loaded:")
print(" - pmids_results.parquet", len(pmid_df))
print(" - pmids_results_filtered.parquet", len(filtered_df))
print(" - pathogen_tiab_counts.parquet", len(pa_counts_df))
print(" - antibiotic_tiab_counts.parquet", len(ab_counts_df))

Loaded:
 - pmids_results.parquet 1893850
 - pmids_results_filtered.parquet 518057
 - pathogen_tiab_counts.parquet 39632
 - antibiotic_tiab_counts.parquet 12520


## Rebuild `pmids_results_positive.parquet`

This uses the corrected combination-level logic so that positive PMIDs are kept only for the reviewed `(pmid, pathogen, antibiotic, start_date, end_date)` rows.

In [3]:
pmids_positive = build_positive_dataset(
    pmid_df,
    filtered_df,
    output_path="pmids_results_positive.parquet",
)

print("Saved pmids_results_positive.parquet", len(pmids_positive))
pmids_positive.head()

Saved pmids_results_positive.parquet 24049


,pathogen,antibiotic,start_date,end_date,pmids,count,venue_histogram
0,Abiotrophia defectiva,Cefepime,2015/01/01,2019/12/31,[27216060],1,"{""Antimicrobial agents and chemotherapy"": 1}"
1,Abiotrophia defectiva,Cefotaxime,2015/01/01,2019/12/31,[27216060],1,"{""Antimicrobial agents and chemotherapy"": 1}"
2,Abiotrophia defectiva,Ceftriaxone,2010/01/01,2014/12/31,[24034902],1,"{""Diagnostic microbiology and infectious disea..."
3,Abiotrophia defectiva,Ciprofloxacin,2010/01/01,2014/12/31,[24034902],1,"{""Diagnostic microbiology and infectious disea..."
4,Abiotrophia defectiva,Clindamycin,2000/01/01,2004/12/31,[10681362],1,"{""Antimicrobial agents and chemotherapy"": 1}"


## Recompute `amr_df_normalised.parquet`

In [4]:
norm_df = normalise_by_background(
    pmids_positive,
    pathogen_year_df=pa_counts_df,
    antibiotic_year_df=ab_counts_df,
    cfg=BackgroundNormaliseConfig(alpha=1.0),
)

norm_df.to_parquet("amr_df_normalised.parquet", index=False)
print("Saved amr_df_normalised.parquet", len(norm_df))
norm_df.head()

Saved amr_df_normalised.parquet 24049


,pathogen,antibiotic,start_date,end_date,pmids,count,venue_histogram,start_year,end_year,bg_pathogen_n,bg_antibiotic_n,frac_among_pathogen,frac_among_antibiotic,geo_frac,expected_bg,resid_log_bg,z_bg
0,Abiotrophia defectiva,Cefepime,2015/01/01,2019/12/31,[27216060],1,"{""Antimicrobial agents and chemotherapy"": 1}",2015,2019,45,1075,0.043478,0.001859,0.008990,1.000000,5.000000e-10,1.000000e-09
1,Abiotrophia defectiva,Cefotaxime,2015/01/01,2019/12/31,[27216060],1,"{""Antimicrobial agents and chemotherapy"": 1}",2015,2019,45,1510,0.043478,0.001324,0.007586,1.000000,5.000000e-10,1.000000e-09
2,Abiotrophia defectiva,Ceftriaxone,2010/01/01,2014/12/31,[24034902],1,"{""Diagnostic microbiology and infectious disea...",2010,2014,24,2233,0.080000,0.000895,0.008463,1.000000,5.000000e-10,1.000000e-09
3,Abiotrophia defectiva,Ciprofloxacin,2010/01/01,2014/12/31,[24034902],1,"{""Diagnostic microbiology and infectious disea...",2010,2014,24,5755,0.080000,0.000347,0.005272,1.000000,5.000000e-10,1.000000e-09
4,Abiotrophia defectiva,Clindamycin,2000/01/01,2004/12/31,[10681362],1,"{""Antimicrobial agents and chemotherapy"": 1}",2000,2004,20,1076,0.095238,0.001857,0.013299,0.336455,4.031267e-01,1.143950e+00


## Sanity check

If any rows remain where `count` is larger than the background totals, the remaining issue is in the already-collected source count files rather than in the rebuild logic above.

In [5]:
bad = norm_df[
    (norm_df["count"] > norm_df["bg_pathogen_n"]) |
    (norm_df["count"] > norm_df["bg_antibiotic_n"])
].copy()

print("remaining violations:", len(bad))

display(
    bad[
        [
            "pathogen",
            "antibiotic",
            "start_date",
            "end_date",
            "count",
            "bg_pathogen_n",
            "bg_antibiotic_n",
        ]
    ].sort_values("count", ascending=False).head(30)
)

remaining violations: 53


,pathogen,antibiotic,start_date,end_date,count,bg_pathogen_n,bg_antibiotic_n
12889,Mycobacterium obuense,Clarithromycin,2020/01/01,2024/12/31,27,12,2579
12571,Mycobacterium flavescens,Clarithromycin,2020/01/01,2024/12/31,22,3,2579
12888,Mycobacterium obuense,Clarithromycin,2015/01/01,2019/12/31,17,11,2296
12570,Mycobacterium flavescens,Clarithromycin,2015/01/01,2019/12/31,14,12,2296
12612,Mycobacterium flavescens,Rifampicin,2015/01/01,2019/12/31,14,12,4156
12575,Mycobacterium flavescens,Doxycycline,2020/01/01,2024/12/31,11,3,5531
12613,Mycobacterium flavescens,Rifampicin,2020/01/01,2024/12/31,10,3,5303
12544,Mycobacterium flavescens,Amikacin,2020/01/01,2024/12/31,10,3,3257
12887,Mycobacterium obuense,Clarithromycin,2010/01/01,2014/12/31,9,3,1850
12564,Mycobacterium flavescens,Ciprofloxacin,2020/01/01,2024/12/31,9,3,10037
